## 1. Import Libraries and Configure Display

In [23]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## 2. Load Source Data into a DataFrame

In [21]:
research_root = Path.cwd()

if not (research_root / "Single Activity Data").exists():
    # Fallback if notebook is opened from a different working directory.
    research_root = Path.cwd() / "Research Data"

rd_files = sorted(research_root.rglob("range_doppler.csv"))

records = []
for csv_path in rd_files:
    try:
        df = pd.read_csv(csv_path)
        if not {"timestamp_s", "frame_number"}.issubset(df.columns):
            # Headerless fallback
            df = pd.read_csv(
                csv_path,
                header=None,
                names=["timestamp_s", "frame_number", "range_bin", "doppler_bin", "signal_strength"],
            )

        df["timestamp_s"] = pd.to_numeric(df["timestamp_s"], errors="coerce")
        df["frame_number"] = pd.to_numeric(df["frame_number"], errors="coerce")
        df = df.dropna(subset=["timestamp_s", "frame_number"])
        if df.empty:
            continue

        rel_parts = csv_path.parent.relative_to(research_root).as_posix().split("/")

        # Required structure:
        # 1st column: Activity Type
        # 2nd column: Activity Name
        # 3rd column: Distance
        if rel_parts and rel_parts[0] == "Single Activity Data":
            activity_type = "Single Activity Data"
            activity_name = rel_parts[1] if len(rel_parts) > 1 else "Unknown"
            distance = rel_parts[2] if len(rel_parts) > 2 else "Unknown"
        elif rel_parts and rel_parts[0] == "Social Behaviour":
            activity_type = "Multi Person Activity Data"
            # Social Behaviour/2 People (update)/<Activity>/<Distance>
            activity_name = rel_parts[2] if len(rel_parts) > 2 else "Unknown"
            distance = rel_parts[3] if len(rel_parts) > 3 else "Unknown"
        else:
            activity_type = "Other"
            activity_name = rel_parts[1] if len(rel_parts) > 1 else "Unknown"
            distance = rel_parts[-1] if rel_parts else "Unknown"

        duration_s = float(df["timestamp_s"].max() - df["timestamp_s"].min())
        frame_count = int(df["frame_number"].nunique())

        records.append(
            {
                "Activity Type": activity_type,
                "Activity Name": activity_name,
                "Distance": distance,
                "Duration (s)": duration_s,
                "Frame Count": frame_count,
            }
        )
    except Exception as exc:
        print(f"Skip {csv_path}: {exc}")

samples_df = pd.DataFrame(records)

# Distance sort key so 2m comes before 2.5m
# Works for labels like: 1m, 2m, 2.5m, 2.5 m, and also plain numeric labels like 1,2,3.
distance_num = pd.to_numeric(
    samples_df["Distance"].astype(str).str.replace("m", "", regex=False).str.strip(),
    errors="coerce",
)
samples_df = samples_df.assign(_distance_order=distance_num)

summary_table = (
    samples_df
    .groupby(["Activity Type", "Activity Name", "Distance", "_distance_order"], as_index=False)
    .agg(
        Total_Duration_s=("Duration (s)", "sum"),
        Total_Frame_Count=("Frame Count", "sum"),
    )
    .sort_values(["Activity Type", "Activity Name", "_distance_order", "Distance"])
    .drop(columns=["_distance_order"])
    .reset_index(drop=True)
)

summary_table["Total_Duration_s"] = summary_table["Total_Duration_s"].round(3)

summary_table

,Activity Type,Activity Name,Distance,Total_Duration_s,Total_Frame_Count
0,Multi Person Activity Data,Approaching,1m,19.000,16
1,Multi Person Activity Data,Approaching,2m,18.188,18
2,Multi Person Activity Data,Approaching,2.5 m,21.994,21
3,Multi Person Activity Data,Approaching,3m,15.993,17
4,Multi Person Activity Data,Approaching,4m,17.000,17
5,Multi Person Activity Data,Approaching,5m,28.008,27
6,Multi Person Activity Data,Sitting,1m,38.992,38
7,Multi Person Activity Data,Sitting,2m,42.200,38
8,Multi Person Activity Data,Sitting,2.5m,52.004,48
9,Multi Person Activity Data,Sitting,3m,52.999,48


In [31]:
# 5. Short summary table with sample counts, activity counts, total duration, and collection notes
import pandas as pd

if samples_df.empty:
    summary_short = pd.DataFrame()
else:
    activity_count_by_type = (
        samples_df.groupby("Activity Type")["Activity Name"]
        .nunique()
        .rename("Activities_Under_Activity_Type")
        .reset_index()
    )

    summary_short = (
        samples_df.groupby(["Activity Type", "Activity Name"], as_index=False)
        .agg(
            Sample_Count=("Distance", "count"),
            Total_Duration_s=("Duration (s)", "sum"),
        )
        .merge(activity_count_by_type, on="Activity Type", how="left")
    )

    def normalize_activity(name: str) -> str:
        return str(name).strip().lower().replace("_", " ")

    collection_notes = {
        "sitting": "Person stands first (8-10)s, then sits down, and remains until end of recording.",
        "standing": "Person sits first (8-10)s, then stands up, and remains standing until end.",
        "approaching": "Person stands a few seconds, walks from random direction toward each labeled distance from sensor(e.g., 1m), then stands for final few seconds.",
        "splitting": "Person stands at each labeled distance (e.g., 1m) briefly, walks away in random direction, then stops and stands for final few seconds.",
        "walking": "Walking in different random directions during each sample.",
        "walking togather": "People have conversation and walk together in different random directions during each sample.",
        "waiving": "Person stands at each labeled point (e.g., 1m, 2m) and waves hands in front of the sensor from start to end of recording.",
    }

    summary_short["Data_Collection_Method"] = summary_short["Activity Name"].apply(
        lambda x: collection_notes.get(
            normalize_activity(x),
            "Recorded using defined protocol.",
        )
    )

    summary_short["Total_Duration_s"] = summary_short["Total_Duration_s"].round(3)
    summary_short = summary_short.sort_values(["Activity Type", "Activity Name"]).reset_index(drop=True)

# Show full text in the DataFrame output (no truncation).
pd.set_option("display.max_colwidth", None)
summary_short

,Activity Type,Activity Name,Sample_Count,Total_Duration_s,Activities_Under_Activity_Type,Data_Collection_Method
0,Multi Person Activity Data,Approaching,6,120.183,5,"Person stands a few seconds, walks from random direction toward each labeled distance from sensor(e.g., 1m), then stands for final few seconds."
1,Multi Person Activity Data,Sitting,6,288.190,5,"Person stands first (8-10)s, then sits down, and remains until end of recording."
2,Multi Person Activity Data,Splitting,6,131.367,5,"Person stands at each labeled distance (e.g., 1m) briefly, walks away in random direction, then stops and stands for final few seconds."
3,Multi Person Activity Data,Standing,6,307.857,5,"Person sits first (8-10)s, then stands up, and remains standing until end."
4,Multi Person Activity Data,Walking_togather,6,404.009,5,People have conversation and walk together in different random directions during each sample.
5,Single Activity Data,Sitting,5,214.643,4,"Person stands first (8-10)s, then sits down, and remains until end of recording."
6,Single Activity Data,Standing,5,245.867,4,"Person sits first (8-10)s, then stands up, and remains standing until end."
7,Single Activity Data,Waiving,5,161.018,4,"Person stands at each labeled point (e.g., 1m, 2m) and waves hands in front of the sensor from start to end of recording."
8,Single Activity Data,Walking,5,232.160,4,Walking in different random directions during each sample.
